# 算法模式

各节为独立示例，可按需阅读和运行。

目录：

- 滑动窗口
- 子集、排列和组合
- 变形二分查找
- Top K Elements
- 二叉树遍历
- 图算法
- 双指针
- 动态规划
- 贪心算法
- 回溯、递归和分治
- 排序算法

## 滑动窗口

滑动窗口适合处理连续子数组或子字符串问题。核心是维护窗口边界，并在窗口不满足约束时移动左边界。

### 无重复字符的最长子串

给定字符串 `s`，返回不含重复字符的最长子串长度。

In [ ]:
def length_of_longest_substring(s: str) -> int:
    start = 0
    max_len = 0
    char_set: set[str] = set()

    for end in range(len(s)):
        while s[end] in char_set:
            char_set.remove(s[start])
            start += 1
        char_set.add(s[end])
        max_len = max(max_len, end - start + 1)

    return max_len


input_string = "abcbbabcbb"
print(length_of_longest_substring(input_string))

## 子集、排列和组合

这类问题关注“从一组元素中生成所有可能结构”。小规模输入可以直接使用标准库，面试题通常需要掌握回溯写法。

### 排列

给定不含重复元素的整数数组 `nums`，返回所有可能的排列。

In [ ]:
from itertools import permutations


def permute(nums: list[int]):
    return list(permutations(nums))


permute([1, 2, 3])

## 变形二分查找

二分查找不只用于普通有序数组，也可以用于旋转数组、边界查找和答案空间查找。关键是每轮明确哪一半仍然有序或仍可能包含答案。

### 在旋转排序数组中搜索

`nums` 是升序数组经过一次旋转后的结果，元素互异，要求返回 `target` 的下标。

In [ ]:
def search(nums: list[int], target: int) -> int:
    left, right = 0, len(nums) - 1

    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid

        if nums[left] <= nums[mid]:
            if nums[left] <= target < nums[mid]:
                right = mid - 1
            else:
                left = mid + 1
        else:
            if nums[mid] < target <= nums[right]:
                left = mid + 1
            else:
                right = mid - 1

    return -1


search([4, 5, 6, 7, 0, 1, 2], 0)

## Top K Elements

Top K 问题常用排序、堆或快速选择。排序实现简单，堆适合只需要少量最大/最小元素的场景。

### 数组中第 K 大元素

In [ ]:
import heapq


def find_kth_largest(nums: list[int], k: int) -> int:
    nums.sort(reverse=True)
    return nums[k - 1]


def find_kth_largest_with_heap(nums: list[int], k: int) -> int:
    return heapq.nlargest(k, nums)[-1]


nums = [3, 2, 1, 5, 6, 4]
print(find_kth_largest(nums[::], 2))
print(find_kth_largest_with_heap(nums, 2))

## 二叉树遍历

二叉树常见遍历包括 DFS 的前序、中序、后序，以及 BFS 的层序遍历。递归写法表达清晰，迭代写法更接近栈和队列的本质。

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.left.left = TreeNode(4)
root.left.right = TreeNode(5)

### DFS

In [ ]:
def preorder_traversal(node):
    if node is not None:
        print(node.val)
        preorder_traversal(node.left)
        preorder_traversal(node.right)


def preorder_traversal_loop(root):
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        result.append(node.val)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return result


def inorder_traversal(node):
    if node is not None:
        inorder_traversal(node.left)
        print(node.val)
        inorder_traversal(node.right)


def inorder_traversal_loop(root):
    result = []
    stack = []
    current = root
    while current or stack:
        while current:
            stack.append(current)
            current = current.left
        current = stack.pop()
        result.append(current.val)
        current = current.right
    return result


def postorder_traversal(node):
    if node is not None:
        postorder_traversal(node.left)
        postorder_traversal(node.right)
        print(node.val)


def postorder_traversal_loop(root):
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        result.append(node.val)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return result[::-1]


print(preorder_traversal_loop(root))
print(inorder_traversal_loop(root))
print(postorder_traversal_loop(root))

### 深度和 BFS

In [ ]:
def tree_depth(node) -> int:
    if node is None:
        return 0
    return max(tree_depth(node.left), tree_depth(node.right)) + 1


def bfs_binary_tree(root):
    queue = [root]
    result = []
    while queue:
        node = queue.pop(0)
        result.append(node.val)
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)
    return result


print(tree_depth(root))
print(bfs_binary_tree(root))

## 图算法

图算法的核心是图表示、遍历顺序和状态记录。这个 notebook 包含 Dijkstra、BFS、DFS、DFS 后序、DAG 判定和拓扑排序。

### 单源最短路径

In [ ]:
def dijkstra(graph, node):
    settled = {node: 0}
    unsettled = dict({point: float("infinity") for point in graph}, **graph.get(node))
    unsettled.pop(node, None)

    while unsettled:
        unsettled = dict(
            sorted(unsettled.items(), key=lambda item: item[1], reverse=True)
        )
        cur_point, cur_dis = unsettled.popitem()
        settled[cur_point] = cur_dis

        for point, dis in graph.get(cur_point).items():
            if unsettled.get(point) and unsettled.get(point) >= dis + cur_dis:
                unsettled[point] = dis + cur_dis

    return settled


graph = {
    "A": {"B": 10, "C": 3},
    "B": {"A": 10, "C": 1, "D": 2},
    "C": {"A": 3, "B": 1, "D": 4},
    "D": {"B": 2, "C": 4},
}

print(dijkstra(graph, "A"))

### BFS 和 DFS

In [ ]:
graph = {
    "A": ["B", "C", "D"],
    "B": ["A", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "H"],
    "F": ["C"],
    "G": ["D"],
    "H": ["E"],
}


def bfs_graph(graph, start):
    result = []
    visited = {start}
    queue = [start]
    while queue:
        point = queue.pop(0)
        result.append(point)
        for next_point in graph.get(point, []):
            if next_point not in visited:
                visited.add(next_point)
                queue.append(next_point)
    return result


def dfs_graph(graph, start):
    result = []
    visited = set()
    stack = [start]
    while stack:
        point = stack.pop()
        if point not in visited:
            visited.add(point)
            result.append(point)
            stack.extend(graph.get(point, []))
    return result


def dfs_postorder_graph(graph, start):
    result = []
    visited = set()

    def dfs(point):
        if point in visited:
            return
        visited.add(point)
        for next_point in graph.get(point, []):
            dfs(next_point)
        result.append(point)

    dfs(start)
    return result


print(bfs_graph(graph, "A"))
print(dfs_graph(graph, "A"))
print(dfs_postorder_graph(graph, "A"))

### 有向图判环 / DAG 判断

In [ ]:
from collections import defaultdict, deque


def is_dag_dfs(graph):
    visited = set()
    path = set()

    def dfs(node):
        if node in path:
            return False
        if node in visited:
            return True

        visited.add(node)
        path.add(node)
        for next_node in graph.get(node, []):
            if not dfs(next_node):
                return False
        path.remove(node)
        return True

    nodes = set(graph)
    for ends in graph.values():
        nodes.update(ends)

    for node in nodes:
        if node not in visited and not dfs(node):
            return False
    return True


def is_dag_bfs(graph):
    indegree = defaultdict(int)
    nodes = set(graph)

    for node, next_nodes in graph.items():
        nodes.add(node)
        for next_node in next_nodes:
            indegree[next_node] += 1
            nodes.add(next_node)

    queue = deque([node for node in nodes if indegree[node] == 0])
    count = 0

    while queue:
        node = queue.popleft()
        count += 1
        for next_node in graph.get(node, []):
            indegree[next_node] -= 1
            if indegree[next_node] == 0:
                queue.append(next_node)

    return count == len(nodes)


acyclic_graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["D"],
    "D": [],
}

cyclic_graph = {
    "A": ["B"],
    "B": ["C"],
    "C": ["A"],
}

print(is_dag_dfs(acyclic_graph))
print(is_dag_dfs(cyclic_graph))
print(is_dag_bfs(acyclic_graph))
print(is_dag_bfs(cyclic_graph))

### 拓扑排序

In [ ]:
def can_finish(prerequisites):
    graph = {}
    for end, start in prerequisites:
        graph[start] = graph.get(start, set()) | {end}

    indegree = {}
    for start, ends in graph.items():
        indegree.setdefault(start, 0)
        for end in ends:
            indegree[end] = indegree.get(end, 0) + 1

    queue = {node for node, value in indegree.items() if value == 0}
    result = []
    while queue:
        node = queue.pop()
        result.append(node)
        for key in graph.get(node, []):
            indegree[key] -= 1
            if indegree[key] == 0:
                queue.add(key)

    return sum(indegree.values()) == 0, result, graph


can_finish([[1, 4], [2, 4], [3, 1], [3, 2]])

## 双指针

双指针常用于有序数组、区间收缩和原地处理问题。核心是每轮移动哪个指针，以及移动后是否仍保持不变量。

### 两数之和

在升序数组中找到两个数，使其和等于目标值。

In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    left = 0
    right = len(nums) - 1

    while left < right:
        current = nums[left] + nums[right]
        if current == target:
            return [left + 1, right + 1]
        if current > target:
            right -= 1
        else:
            left += 1

    return []


two_sum([-1000, -1, 0, 1], 1)

## 动态规划

动态规划适合有重叠子问题和最优子结构的问题。关键是定义状态、转移方程、初始化和遍历顺序。

### 最长公共子序列和最长公共子串

In [ ]:
def longest_common_subsequence(left: str, right: str) -> str:
    rows = len(left) + 1
    cols = len(right) + 1
    matrix = [[0] * cols for _ in range(rows)]

    for row in range(1, rows):
        for col in range(1, cols):
            if left[row - 1] == right[col - 1]:
                matrix[row][col] = matrix[row - 1][col - 1] + 1
            else:
                matrix[row][col] = max(matrix[row][col - 1], matrix[row - 1][col])

    seq = []
    row, col = rows - 1, cols - 1
    while row > 0 and col > 0:
        if left[row - 1] == right[col - 1]:
            seq.append(left[row - 1])
            row -= 1
            col -= 1
        elif matrix[row - 1][col] >= matrix[row][col - 1]:
            row -= 1
        else:
            col -= 1
    return "".join(reversed(seq))


def longest_common_substring(left: str, right: str) -> str:
    rows = len(left)
    cols = len(right)
    matrix = [[0] * (cols + 1) for _ in range(rows + 1)]
    end = 0
    max_len = 0

    for row in range(1, rows + 1):
        for col in range(1, cols + 1):
            if left[row - 1] == right[col - 1]:
                matrix[row][col] = matrix[row - 1][col - 1] + 1
                if matrix[row][col] > max_len:
                    max_len = matrix[row][col]
                    end = row

    return left[end - max_len : end]


print(longest_common_subsequence("ABEFF", "ADEOFOOBP"))
print(longest_common_substring("ABCBDCBABPLB", "BDCABPLL"))

### 不同路径和爬楼梯

In [ ]:
def unique_paths_with_obstacles(obstacle_grid: list[list[int]]) -> int:
    rows = len(obstacle_grid)
    cols = len(obstacle_grid[0])
    matrix = [[0] * (cols + 1) for _ in range(rows + 1)]
    matrix[1][1] = 1

    for row in range(1, rows + 1):
        for col in range(1, cols + 1):
            if obstacle_grid[row - 1][col - 1]:
                matrix[row][col] = 0
            else:
                matrix[row][col] += matrix[row - 1][col] + matrix[row][col - 1]

    return matrix[-1][-1]


def climb_stairs(num: int) -> int:
    previous, current = 1, 1
    for _ in range(3, num + 2):
        previous, current = current, previous + current
    return current


print(unique_paths_with_obstacles([[0, 0], [0, 0]]))
print(climb_stairs(4))

## 贪心算法

贪心算法每一步都做局部最优选择。使用前需要确认局部最优能推出全局最优。

### 最少主持人数

区间调度类问题常用排序和堆维护当前占用资源。

In [ ]:
import heapq


def minimum_num_of_hosts(intervals: list[list[int]]) -> int:
    intervals = sorted(intervals, key=lambda item: (item[0], item[1]))
    hosts = [-float("inf")]

    for start, end in intervals:
        if hosts[0] <= start:
            heapq.heappop(hosts)
        heapq.heappush(hosts, end)

    return len(hosts)


nums = [[547, 612], [417, 551], [132, 132], [168, 446], [95, 747]]
minimum_num_of_hosts(nums)

## 回溯、递归和分治

回溯适合搜索所有可行解；递归和分治适合把问题拆成同构子问题。这个 notebook 保留排列和路径搜索示例。

In [ ]:
from itertools import permutations


def permute_unique(nums):
    return sorted(set(permutations(nums)))


def permute_string_with_itertools(text: str) -> list[str]:
    return ["".join(item) for item in permutations(text)]


def permute_string(text: str) -> list[str]:
    cur_idx = 0
    cur_substrs = [text]
    next_substrs = []

    while cur_idx < len(text) - 1:
        substr = cur_substrs.pop()
        for idx in range(cur_idx, len(text)):
            new_text = list(substr)
            new_text[idx], new_text[cur_idx] = new_text[cur_idx], new_text[idx]
            next_substrs.append("".join(new_text))
        if not cur_substrs:
            cur_substrs = next_substrs
            next_substrs = []
            cur_idx += 1

    return sorted(set(cur_substrs))


print(permute_unique([1, 1, 2]))
print(permute_string_with_itertools("ABCD"))
print(permute_string("ABCD"))

In [ ]:
def unique_paths_backtracking(obstacle_grid: list[list[int]]) -> int:
    result = 0
    stack = [(0, 0)]

    while stack:
        x, y = stack.pop()
        row = len(obstacle_grid)
        col = len(obstacle_grid[x])
        if obstacle_grid[x][y]:
            continue
        if x == row - 1 and y == col - 1:
            result += 1
        if x + 1 < row:
            stack.append((x + 1, y))
        if y + 1 < col:
            stack.append((x, y + 1))

    return result


unique_paths_backtracking([[0]])

## 排序算法

这个 notebook 汇总常见排序算法。基于比较的排序算法在最坏情况下的时间复杂度下界为 $\Omega(n \log n)$；在输入满足特定约束时，非比较排序可以达到线性时间复杂度。

In [ ]:
def bubble_sort(nums):
    nums = nums[::]
    for i in range(len(nums) - 1):
        swapped = False
        for j in range(len(nums) - 1 - i):
            if nums[j] > nums[j + 1]:
                nums[j], nums[j + 1] = nums[j + 1], nums[j]
                swapped = True
        if not swapped:
            break
    return nums


def selection_sort(nums):
    nums = nums[::]
    for i in range(len(nums) - 1):
        min_idx = i
        for j in range(i + 1, len(nums)):
            if nums[j] < nums[min_idx]:
                min_idx = j
        nums[i], nums[min_idx] = nums[min_idx], nums[i]
    return nums


def insertion_sort(nums):
    nums = nums[::]
    for i in range(1, len(nums)):
        base = nums[i]
        while i > 0 and base <= nums[i - 1]:
            nums[i] = nums[i - 1]
            i -= 1
        nums[i] = base
    return nums


sample = [1, 3, 32, 4, 1, -100, 23, 17, 23, 6, -7, 2, 9, -1000]
print(bubble_sort(sample))
print(selection_sort(sample))
print(insertion_sort(sample))

In [ ]:
def quick_sort(nums):
    nums = nums[::]

    def partition(left, right):
        i, j = left, right
        while i < j:
            while i < j and nums[j] >= nums[left]:
                j -= 1
            while i < j and nums[i] <= nums[left]:
                i += 1
            nums[i], nums[j] = nums[j], nums[i]
        nums[left], nums[i] = nums[i], nums[left]
        return i

    def sort(left, right):
        if left >= right:
            return
        idx = partition(left, right)
        sort(left, idx - 1)
        sort(idx + 1, right)

    sort(0, len(nums) - 1)
    return nums


def merge_sort(nums):
    nums = nums[::]

    def merge(left, mid, right):
        i, j = left, mid + 1
        tmp_array = []
        while i <= mid and j <= right:
            if nums[i] <= nums[j]:
                tmp_array.append(nums[i])
                i += 1
            else:
                tmp_array.append(nums[j])
                j += 1
        tmp_array.extend(nums[i : mid + 1] if i <= mid else nums[j : right + 1])
        nums[left : right + 1] = tmp_array

    def sort(left, right):
        if left >= right:
            return
        mid = (right + left) // 2
        sort(left, mid)
        sort(mid + 1, right)
        merge(left, mid, right)

    sort(0, len(nums) - 1)
    return nums


print(quick_sort(sample))
print(merge_sort(sample))

In [ ]:
def heap_sort(nums):
    nums = nums[::]

    def sift_down(n, i):
        while True:
            max_idx = i
            left = 2 * i + 1
            right = 2 * i + 2
            if left < n and nums[left] > nums[max_idx]:
                max_idx = left
            if right < n and nums[right] > nums[max_idx]:
                max_idx = right
            if max_idx == i:
                break
            nums[max_idx], nums[i] = nums[i], nums[max_idx]
            i = max_idx

    n = len(nums)
    for i in range(n // 2 - 1, -1, -1):
        sift_down(n, i)
    for i in range(n - 1, -1, -1):
        nums[0], nums[i] = nums[i], nums[0]
        sift_down(i, 0)
    return nums


print(heap_sort(sample))

In [ ]:
def counting_sort(nums):
    if not nums:
        return []
    min_num = min(nums)
    max_num = max(nums)
    counts = [0] * (max_num - min_num + 1)
    result = []
    for num in nums:
        counts[num - min_num] += 1
    for idx, count in enumerate(counts):
        result.extend([idx + min_num] * count)
    return result


def radix_sort_non_negative(nums):
    if not nums:
        return []
    max_num = max(nums)
    base = 1
    while base <= max_num:
        buckets = [[] for _ in range(10)]
        for num in nums:
            buckets[num // base % 10].append(num)
        nums = [num for bucket in buckets for num in bucket]
        base *= 10
    return nums


def radix_sort(nums):
    negatives = [-num for num in nums if num < 0]
    positives = [num for num in nums if num >= 0]
    return [
        -num for num in reversed(radix_sort_non_negative(negatives))
    ] + radix_sort_non_negative(positives)


print(counting_sort(sample))
print(radix_sort(sample))